# Multi-Asset Market Risk Engine
### Quantitative Risk Analysis & Financial Engineering Portfolio Project

**Author:** Tawanda Magwenzi
**Framework:** Parametric, Historical, and Monte Carlo Value at Risk (VaR)

---

## 1. Project Objective & Core Framework
This notebook implements an automated market risk management engine to evaluate
1. **Parametric (Variance-Covariance) VaR**
2. **Historical Simulation VaR**
3. **Monte Carlo Simulation VaR**

The engine calculates both **Value at Risk (VaR)** and **Expected Shortfall (ES)**

## 2. Comprehensive Data Description & Ingestion Methodology

To construct a mathematically sound risk engine, the data pipeline pulls two distinct types of data streams across a coordinated historical temporary window:

### A. Asset Price Ingestion (Yahoo Finance API)
*   **Data Source:** 'yfinance' open-source financial gateway.
*   **Data Attribute Used:** 'Adj Close' (Adjusted Daily Closing Price). This is structurally mandatory over the standard 'Close' price because it accounts for external corporate actions, stock splits and dividend distributions, preserving the continuous return matrix.
*   **Time Series Horizon:** A 3-year historical baseline window.
*   **Target Portfolio Assets:**
    *   'AAPL' (Apple Inc. - High Beta Growth Equity)
    *   'MSFT' (Microsoft Corp. - Mega-cap Tech Stability Equity)
    *   'SPY' (SPDR S&P 500 ETF Trust - Broad Market Benchmark Index)

### B. Risk-Free Ingestion (St. Lous FRED API)
*   **Data Source:** 'pandas_datareader' targeting the Federal Reserve Ecenomic Data (FRED) database.
*   **Data Attribute Used:** 'TB3MS' (3-Month US Treasury Bill Secondary Market Rate).
*   **Transformation:** Represents the macro risk-free proxy. The raw annualized percentage yield converted to a localized daily decimal format (\(\frac{\text{Yield}}{100} / 252\)) to feed downstream option pricing and stochastic drift calculations.

### C. Mathematical Property: Invariant Transformation (Log Returns)
Raw asset price series are non-stationary and cannot be sampled directly for stochastic pricing models due to long-term drift. We apply a transcendental logarithmic transformation to enforce stationarity:

\[r_t = \ln\left(\frac{P_t}{P_{t-1}}\right)\]

Where \(P_t\) is the adjusted price at day t. Log returns are chosen over simple arithmetic returns because they provide **time additivity** (\(\sum r_t\)) and strictly map to the normal distribution parameters required by Geometric Brownian Motion simulations.

In [1]:
# Import core quantitative and data pipeline packages
import os
import logging
from typing import List, Tuple
import pandas as pd
import numpy as np
import yfinance as yf
import pandas_datareader.data as web
import matplotlib.pyplot as plt

# Configure clean logging indicators inside the Jupyter layout
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger("RiskEngine")

print("Packages loaded successfully. Computational environment ready.")

ModuleNotFoundError: No module named 'pandas'

In [2]:
class DataAcquisitionLayer:
    """ 
    Handles robust fetching of historical market data and risk-free rates,
    computing mathematical log returns for the market risk simulation engine.
    """
    def __init__(self, start_date: str, start_end: str):
        self.start_date = start_date
        self.start_end = start_end

    def fetch_equity_prices(self, tickers: List[str]) -> pd.DataFrame:
        """Fetches adjusted daily closing prices from Yahoo Finance."""
        logger.info(f"Pulling equity prices for: {tickers}")
        try:
            data = yf.download(tickers, start=self.start_date, end=self.end_date)
            adj_close = data['Adj Close'] if 'Adj Close' in data.columns else data['Close']

            if isinstance(adj_close, pd.Series):
                adj_close = adj_close.to_frame(name=tickers)

            return adj_close.dropna(how='all').ffill().bfill()
        except Exception as e:
            logger.error(f"Failed to fetch equity prices: {e}")
            raise e
        
    def fetch_risk_free_rate(self, series_id: str = "TB3MS") -> pd.Series:
        """Fetches historical 3-Month Treasury Bill rate from FRED as a daily decimal"""
        logger.info(f"Pulling macroeconomic benchmark from FRED: {series_id}")
        try:
            df = web.DataReader(series_id, 'fred', self.start_date, self.end_date)
            daily_rf_decimal = (df[series_id] / 100) / 252
            return daily_rf_decimal
        except Exception as e:
            logger.error(f"Failed to fetch FRED data: {e}")
            raise e
    

    @staticmethod
    def calculate_log_returns(prices_df: pd.DataFrame) -> pd.DataFrame:
        """Transforms raw price time-series into stationary daily log returns."""
        log_returns = np.log(prices_df / prices_df.shift(1))
        return log_returns.dropna(how='all')
    
    def build_risk_matrix(self, tickers: List[str]) -> Tuple[pd.DataFrame, pd.DataFrame]:
        """Orchestrates data layer pipeline."""
        prices = self.fetch_equity_prices(tickers)
        log_returns = self.calculate_log_returns(prices)
        return prices, log_returns

In [ ]:
# Instantiate the data pipeline layer across a 3-year historical window
pipeline = DataAcquisitionLayer(start_date="2023-01-01", start_end="2026-01-01")
portfolio_tickers = ["AAPL", "MSFT", "SPY"]

#Execute price pull and log transformations
raw_prices, clean_returns = pipeline.build_risk_matrix(portfolio_tickers)
daily_rf = pipeline.fetch_risk_free_rate()

# Verification Block: Output descriptive metadata to confirm data integrity
print("\n" + "="*35 + " DATA INTEGRITY VERIFICATION " + "="*35)
print(f"Historical Time Series Length : {raw_price.shape[0]} trading days")
print(f"Asset Invariants (Log Returns): Shape {clean_returns.shape}")
print("\n--- Correlation Matrix (Empirical Co=movements) ---")
print(clean_returns.corr())

#Generate a visual baseline check of the stationary returns distribution
plt.figure(figsize=(12, 6))
for ticker in portfolio_tickers:
    plt.plot(clean_returns[ticker], label=ticker, alpha=0.6)
plt.title("Stationary Daily Log Returns Time Series ($\ln(P_t / P_{t-1})$)")
plt.xlabel("Timeline")
plt.ylabel("Log Scale Shift Percentage")
plt.legend(loc="upper left")
plt.grid(True, linestyle="--", alpha=0.5)
plt.show()